# 5. Annotate prepared windows with local Qwen

Load the interim prepared-window manifests, run the local MLX Qwen3-VL model, and write two joinable outputs per video under `data/processed/<video-name>/`: `window_event_annotations.json` and `yolo_spatial_coordinates.json`. The coordinate file retains full frame-level YOLO `box_xyxy` values only for track IDs referenced by events.

**Rerun behaviour** is controlled by `OVERWRITE`:

- `OVERWRITE = False` (default): windows that were already annotated are reused and the model only runs on missing windows, so an interrupted run resumes where it stopped. With `RETRY_FAILED_WINDOWS = True`, windows whose previous attempt recorded an `error` are retried. If a video's `prepared_windows.json` is newer than its annotations (notebook 4 was rerun), that video is annotated again from scratch.
- `OVERWRITE = True`: every window of every video is annotated again.
- `VIDEO_FILTER` (default `None`) restricts either mode to videos whose name contains the text, so a prompt change can be piloted on one camera with `VIDEO_FILTER = "G328"` and `OVERWRITE = True` without touching the other videos. The `index.json` catalogue is still rebuilt from everything in `data/processed/`.

A video is only kept when it produced **both** halves of the join: at least one window describing an event, and at least one YOLO detection behind those events. A video with neither is erased from `data/processed/` instead of leaving empty `[]` files behind, so stages 6 and 7 never see a video with nothing in it. The trade-off is that such a video has no checkpoint to resume from and is annotated again on the next run.

After every video, the list of stored videos is published to `data/processed/index.json`, which `reports/annotation_explorer.html` reads to populate its video catalogue.

Both output files are checkpointed atomically after each window. The model is only loaded if at least one window needs annotating. When run from `00_run_annotation_pipeline.ipynb`, the runner's `OVERWRITE` takes precedence.

**Model.** `MODEL_ID` selects the MLX vision model; the 8B and 4B Qwen3-VL builds are both listed so a comparison is a one-line change. `VIDEO_FILTER` takes a full video name for one video, a camera id for a camera, or `None` for everything.

**Output budget and actions** (`videometa>=0.0.17`): `max_tokens` is 1500. The earlier 600 cut the JSON mid-string on about 10% of windows, and every one of those windows was lost; the package now also salvages the complete events before a cut. Each event carries an `actions` list drawn from the package's plain-English checklist (`opens a vehicle door`, `talks to another person`, `vehicle reverses`, ...) plus `other_actions` for anything outside it, and the prompt asks for one event per subject per action instead of one summary per window. When `prepared_windows.json` records a `crop_box`, it is passed on so the prompt can say the video is a crop.

This notebook must run natively on Apple Silicon with `mlx-vlm` installed; MLX/Metal is not available in the Airflow Docker container.

In [1]:
import json
import os
from pathlib import Path

import mlx.core as mx
from videometa import LocalQwenEventAnnotator, PreparedWindowInput

# True: re-annotate every window of every video.
# False: reuse windows already annotated and only run the model on the missing ones.
OVERWRITE = True  # Pilot: re-annotate the filtered video with the subject-focus prompt. Set to False afterwards.
# When OVERWRITE is False, also retry windows whose previous attempt ended with an error.
RETRY_FAILED_WINDOWS = True
# Only annotate videos whose name contains this text (e.g. "G328" for one camera). None: every video.
# Use it with OVERWRITE = True to pilot a prompt change on a few videos without re-running the rest.
VIDEO_FILTER = None  # Pilot: one video. A camera id such as "G421" selects a camera; None selects all videos.

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Vision model. The 4B model described windows as "person walks across the car park" and never named a
# door, phone or turn on the G328 pilot, so the 8B model is being tried on the same video. ~5 GB of weights.
MODEL_ID = "mlx-community/Qwen3-VL-8B-Instruct-4bit"
# MODEL_ID = "mlx-community/Qwen3-VL-4B-Instruct-4bit"   # previous model, for a like-for-like comparison

mx.set_cache_limit(2 * 1024**3)
annotator = None  # Loaded lazily, only when a window actually needs annotating.


def get_annotator():
    global annotator
    if annotator is None:
        print(f"Loading local model: {MODEL_ID}", flush=True)
        annotator = LocalQwenEventAnnotator(
            model_id=MODEL_ID,
            video_fps=2.0,             # must match annotated_video_fps in notebook 04
            max_tokens=1500,           # was 600, which truncated ~10% of windows mid-JSON
            # The guidance text alone is ~4,100 tokens after the focus, movement and self-review rules,
            # so 6,000 left room for only 2 sampled frames of spatial features; 7,000 keeps 4.
            prompt_token_budget=7000,
            feature_frames=8,
        )
    return annotator


def read_json(path):
    """Return parsed JSON, or None when the file is missing or was left half-written."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None


def write_json(path, payload):
    """Write JSON atomically so an interrupted run never leaves a corrupt checkpoint."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".part")
    temporary_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temporary_path.replace(path)


def window_key(item):
    return (float(item["start_seconds"]), float(item["end_seconds"]))


# Windows the model returned no events for are dropped from both outputs, so the
# annotation files only contain windows that actually describe something.
# A *failed* window is kept: its "error" key is the only record that the call
# broke, and discarding it would make a broken run look like a quiet video.
DROP_EMPTY_WINDOWS = True


def ordered_results(completed, items):
    """Completed window results in manifest order, empties optionally removed."""
    ordered = [
        completed[window_key(entry)] for entry in items if window_key(entry) in completed
    ]
    if not DROP_EMPTY_WINDOWS:
        return ordered
    return [pair for pair in ordered if pair[0].get("events") or pair[0].get("error")]


# A video is only stored when it produced both halves of the join: at least one
# window describing an event, and at least one YOLO detection behind those events.
# Anything less cannot be explored or exported downstream, so the video is dropped
# entirely instead of leaving empty `[]` files behind in data/processed/.
def is_storable(ordered):
    """True when the video has both event annotations and spatial coordinates."""
    has_events = any(result.get("events") for result, _ in ordered)
    has_detections = any(spatial.get("detections") for _, spatial in ordered)
    return has_events and has_detections


def discard_video_outputs(output_dir):
    """Remove a video's processed outputs, and the directory once it is empty."""
    if not output_dir.is_dir():
        return
    for path in output_dir.iterdir():
        if path.is_file():
            path.unlink()
    try:
        output_dir.rmdir()
    except OSError:
        pass  # Something else lives here; leave the directory alone.


def load_previous_windows(manifest_path, output_path, spatial_output_path):
    """Return reusable {window_key: (result, spatial)} from an earlier run, or {} when stale."""
    if OVERWRITE:
        return {}
    if output_path.is_file() and output_path.stat().st_mtime < manifest_path.stat().st_mtime:
        print("  prepared_windows.json is newer than the annotations; re-annotating all windows")
        return {}
    results = read_json(output_path) or []
    spatial_by_key = {window_key(item): item for item in read_json(spatial_output_path) or []}
    previous = {}
    for result in results:
        key = window_key(result)
        if key not in spatial_by_key:
            continue  # Annotation without its coordinates: redo the window.
        if RETRY_FAILED_WINDOWS and result.get("error"):
            continue
        previous[key] = (result, spatial_by_key[key])
    return previous


def video_directory_names(processed_dir):
    """Names of the video directories holding a complete annotation/coordinate pair."""
    return [
        path.name
        for path in processed_dir.iterdir()
        if path.is_dir()
        and (path / "window_event_annotations.json").is_file()
        and (path / "yolo_spatial_coordinates.json").is_file()
    ]


stored_videos = []
skipped_videos = []
manifest_paths = sorted(INTERIM_DIR.glob("*/prepared_windows.json"))
if VIDEO_FILTER:
    manifest_paths = [path for path in manifest_paths if VIDEO_FILTER in path.parent.name]
if not manifest_paths:
    raise FileNotFoundError(f"No prepared window manifests found in {INTERIM_DIR} matching {VIDEO_FILTER!r}")
print("=" * 72, flush=True)
print("Stage 5: annotating prepared windows with local Qwen", flush=True)
print(
    f"OVERWRITE={OVERWRITE}, RETRY_FAILED_WINDOWS={RETRY_FAILED_WINDOWS}, VIDEO_FILTER={VIDEO_FILTER!r}: "
    f"{len(manifest_paths)} videos found.",
    flush=True,
)
print("=" * 72, flush=True)

for manifest_path in manifest_paths:
    video_name = manifest_path.parent.name
    items = json.loads(manifest_path.read_text(encoding="utf-8"))
    output_dir = PROCESSED_DIR / video_name
    output_path = output_dir / "window_event_annotations.json"
    spatial_output_path = output_dir / "yolo_spatial_coordinates.json"

    completed = load_previous_windows(manifest_path, output_path, spatial_output_path)
    completed = {key: value for key, value in completed.items() if key in {window_key(item) for item in items}}
    pending = [item for item in items if window_key(item) not in completed]
    if not pending and output_path.is_file() and spatial_output_path.is_file():
        stored_videos.append(video_name)
        print(f"Skipping {video_name}: all {len(items)} windows already annotated")
        continue
    print(f"{video_name}: {len(completed)} windows reused, {len(pending)} to annotate")

    model = get_annotator() if pending else None
    for window_index, item in enumerate(pending, start=1):
        print(
            f"  Window {window_index}/{len(pending)}: "
            f"{item['start_seconds']:.1f}s–{item['end_seconds']:.1f}s",
            flush=True,
        )
        prepared = PreparedWindowInput(
            start_seconds=item["start_seconds"],
            end_seconds=item["end_seconds"],
            object_features=tuple(item["object_features"]),
            frame_features=tuple(item["frame_features"]),
            image_messages=tuple(item.get("image_messages", [])),
            artifact_directory=item.get("artifact_directory"),
            annotated_video_path=item.get("annotated_video_path"),
            crop_box=tuple(item["crop_box"]) if item.get("crop_box") else None,
        )
        try:
            events = model.annotate(prepared)
            result = {
                "start_seconds": prepared.start_seconds,
                "end_seconds": prepared.end_seconds,
                "annotated_video_path": prepared.annotated_video_path,
                "events": events,
            }
        except Exception as error:
            events = []
            result = {
                "start_seconds": prepared.start_seconds,
                "end_seconds": prepared.end_seconds,
                "annotated_video_path": prepared.annotated_video_path,
                "events": [],
                "error": str(error),
            }

        event_object_ids = {
            str(involved_object["id"])
            for event in events
            for involved_object in event.get("involved_objects", [])
            if involved_object.get("id") is not None
        }
        detections = []
        for frame in prepared.frame_features:
            for detection in frame.get("detections", []):
                if str(detection.get("track_id")) in event_object_ids:
                    detections.append(
                        {
                            "object_id": str(detection["track_id"]),
                            "frame_index": frame["frame_index"],
                            "timestamp_seconds": frame["timestamp_seconds"],
                            "label": detection["label"],
                            "confidence": detection["confidence"],
                            "box_xyxy": detection["box_xyxy"],
                            "spatial_position": detection["spatial_position"],
                        }
                    )
        spatial_result = {
            "start_seconds": prepared.start_seconds,
            "end_seconds": prepared.end_seconds,
            "event_object_ids": sorted(event_object_ids),
            "detections": detections,
        }
        completed[window_key(item)] = (result, spatial_result)

        # Checkpoint after every window, in manifest order, so an interrupted run can resume.
        ordered = ordered_results(completed, items)
        write_json(output_path, [result for result, _ in ordered])
        write_json(spatial_output_path, [spatial for _, spatial in ordered])

    ordered = ordered_results(completed, items)
    failed = sum(1 for result, _ in completed.values() if result.get("error"))
    empty = sum(
        1 for result, _ in completed.values()
        if not result.get("events") and not result.get("error")
    )

    if not is_storable(ordered):
        discard_video_outputs(output_dir)
        skipped_videos.append(video_name)
        print(
            f"Skipping {video_name}: no event annotations with spatial coordinates "
            f"({empty} windows with no events, {failed} with errors). "
            f"Nothing written to {output_dir}"
        )
        if failed:
            print(f"  warning: {failed} window(s) failed; rerun this video to retry them")
        continue

    # Rewrite both files from the final window set, so a video whose last window was
    # reused (nothing pending) is still written out completely.
    write_json(output_path, [result for result, _ in ordered])
    write_json(spatial_output_path, [spatial for _, spatial in ordered])
    stored_videos.append(video_name)
    print(
        f"Wrote {len(ordered)} window annotations to {output_path} "
        f"({failed} with errors, {empty} with no events "
        f"{'dropped' if DROP_EMPTY_WINDOWS else 'kept'})"
    )
    print(f"Wrote joinable YOLO coordinates to {spatial_output_path}")

# The explorer UI is a static page and cannot list a directory, so publish the
# catalogue of stored videos it should offer.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
write_json(
    PROCESSED_DIR / "index.json",
    {"videos": sorted(video_directory_names(PROCESSED_DIR))},
)
print(
    f"\n{len(stored_videos)} video(s) stored, {len(skipped_videos)} skipped for having "
    f"no annotations or no spatial coordinates."
)
if skipped_videos:
    print("Skipped: " + ", ".join(skipped_videos))
print(f"Wrote the explorer catalogue to {PROCESSED_DIR / 'index.json'}")
print("Done.")

Stage 5: annotating prepared windows with local Qwen
OVERWRITE=True, RETRY_FAILED_WINDOWS=True, VIDEO_FILTER=None: 72 videos found.
2018-03-05.13-15-00.13-20-00.bus.G340: 0 windows reused, 18 to annotate
Loading local model: mlx-community/Qwen3-VL-8B-Instruct-4bit


/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 2833.48it/s]


  Window 1/18: 24.0s–34.0s
  Window 2/18: 32.0s–42.0s
  Window 3/18: 56.0s–66.0s
  Window 4/18: 64.0s–74.0s
  Window 5/18: 72.0s–82.0s
  Window 6/18: 80.0s–90.0s
  Window 7/18: 88.0s–98.0s
  Window 8/18: 96.0s–106.0s
  Window 9/18: 104.0s–114.0s
  Window 10/18: 112.0s–122.0s
  Window 11/18: 120.0s–130.0s
  Window 12/18: 144.0s–154.0s
  Window 13/18: 152.0s–162.0s
  Window 14/18: 176.0s–186.0s
  Window 15/18: 240.0s–250.0s
  Window 16/18: 248.0s–258.0s
  Window 17/18: 280.0s–290.0s
  Window 18/18: 288.0s–298.0s
Wrote 18 window annotations to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G340/window_event_annotations.json (0 with errors, 0 with no events dropped)
Wrote joinable YOLO coordinates to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G340/yolo_spatial_coordinates.json
2018-03-05.13-15-00.13-20-00.bus.G505: 0 windows reused, 10 to annotate
  Window 1

Model output was truncated; salvaged 10 complete events


  Window 3/38: 16.0s–26.0s
  Window 4/38: 24.0s–34.0s
  Window 5/38: 32.0s–42.0s
  Window 6/38: 40.0s–50.0s
  Window 7/38: 48.0s–58.0s
  Window 8/38: 56.0s–66.0s
  Window 9/38: 64.0s–74.0s
  Window 10/38: 72.0s–82.0s
  Window 11/38: 80.0s–90.0s
  Window 12/38: 88.0s–98.0s
  Window 13/38: 96.0s–106.0s


...

  Window 38/38: 296.0s–300.1s


Window prompt was 7635 tokens; retrying with 4 sampled frames


Wrote 38 window annotations to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-07.16-50-00.16-55-00.school.G421/window_event_annotations.json (4 with errors, 0 with no events dropped)
Wrote joinable YOLO coordinates to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-07.16-50-00.16-55-00.school.G421/yolo_spatial_coordinates.json
2018-03-07.16-50-00.16-55-00.school.G638: 0 windows reused, 26 to annotate
  Window 1/26: 0.0s–10.0s
  Window 2/26: 8.0s–18.0s
  Window 3/26: 24.0s–34.0s
  Window 4/26: 32.0s–42.0s
  Window 5/26: 40.0s–50.0s
  Window 6/26: 48.0s–58.0s
  Window 7/26: 56.0s–66.0s
  Window 8/26: 64.0s–74.0s
  Window 9/26: 72.0s–82.0s
  Window 10/26: 80.0s–90.0s
  Window 11/26: 88.0s–98.0s
  Window 12/26: 96.0s–106.0s
  Window 13/26: 104.0s–114.0s
  Window 14/26: 112.0s–122.0s
  Window 15/26: 120.0s–130.0s


Model output was truncated; salvaged 9 complete events


  Window 16/26: 128.0s–138.0s
  Window 17/26: 136.0s–146.0s
  Window 18/26: 184.0s–194.0s
  Window 19/26: 224.0s–234.0s
  Window 20/26: 232.0s–242.0s
  Window 21/26: 240.0s–250.0s
  Window 22/26: 248.0s–258.0s
  Window 23/26: 256.0s–266.0s
  Window 24/26: 272.0s–282.0s
  Window 25/26: 288.0s–298.0s
  Window 26/26: 296.0s–300.0s
Wrote 26 window annotations to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-07.16-50-00.16-55-00.school.G638/window_event_annotations.json (0 with errors, 0 with no events dropped)
Wrote joinable YOLO coordinates to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-07.16-50-00.16-55-00.school.G638/yolo_spatial_coordinates.json
2018-03-07.16-50-01.16-55-01.admin.G326: 0 windows reused, 6 to annotate
  Window 1/6: 80.0s–90.0s
  Window 2/6: 88.0s–98.0s
  Window 3/6: 168.0s–178.0s
  Window 4/6: 176.0s–186.0s
  Window 5/6: 288.0s–298.0s
  Window 6/6: 296.0s–300.0s
Wrote 6 window annotation